In [27]:
from trec_eval import EvalFunction, THE_TOPICS, eval_rerank

name = '/home/mila/l/le.zhang/scratch/DeepRerank/data/combined_qrels.txt'
qrels_dict = EvalFunction.load_qrels(THE_TOPICS.get(name, name))
from datasets import load_dataset
run_file = load_dataset("parquet", data_files='/home/mila/l/le.zhang/scratch/DeepRerank/data/filtered_train.parquet')["train"]




In [35]:
import numpy as np
# 不需要 pytrec_eval 库来读取文件了，因为数据已经是 dict 格式
# from collections import defaultdict # 也不需要 defaultdict 了，因为 Qrels 是 pytrec_eval dict 格式

def calculate_ndcg_runfile_normalized_dict_auto_qid(run_data_dict, qrels_data_dict, k=10):
    """
    计算给定 Run 数据字典的 NDCG@k，归一化项为 Run 数据中文档的理论最高 DCG@k。
    Query ID 从 run_data_dict['hits'][0]['qid'] 中获取。

    Args:
        run_data_dict (dict): 包含 Run 结果的字典，格式为 {'query': '...', 'hits': [...]}.
                              hits 是一个列表，每个元素是 {'docid': '...', 'qid': ..., 'rank': ..., 'score': ..., ...}。
                              假设 hits 中的所有条目都属于同一个 Query。
        qrels_data_dict (dict): pytrec_eval 格式的 Qrels 字典，格式为 {qid: {docid: relevance, ...}, ...}。
        k (int): 计算的截断位置 (e.g., @10)。

    Returns:
        float: 计算得到的 NDCG@k 值。如果无法计算（无 Qrels 判断, Run 数据为空等），返回 0.0。
    """
    all_hits = run_data_dict.get('hits', [])

    if not all_hits:
        # print("Warning: Run data 'hits' list is empty.")
        return 0.0

    # 从第一个 hit 中获取 Query ID
    query_id = all_hits[0].get('qid')
    if query_id is None:
         # print("Warning: 'qid' not found in the first hit.")
         return 0.0

    # 1. 从 Qrels 字典中获取目标 Query 的相关性判断
    query_qrels = qrels_data_dict.get(str(query_id))

    if not query_qrels:
        # print(f"Warning: No relevance judgments found for query {query_id} in Qrels dict.")
        return 0.0

    # 提取 Run 数据中所有文档的 ID，用于计算 IDCG_RunFileDocs
    run_docs_set = {hit.get('docid') for hit in all_hits if hit.get('docid')}

    print(f" run_docs_set: {run_docs_set}")
    # --- 计算 DCG@k (基于 Run 数据实际排序的前 k 个) ---
    dcg = 0.0
    # 取 hits 列表中的前 k 个 (假设已按 score/rank 排序)
    ranked_hits_for_dcg = all_hits[:k]

    for i in range(len(ranked_hits_for_dcg)):
        hit = ranked_hits_for_dcg[i]
        doc_id = hit.get('docid')
        rank = i + 1 # 位置从 1 开始，使用列表索引+1更通用

        if doc_id:
            # 查找文档在 Qrels 中的真实相关性
            relevance = query_qrels.get(doc_id, 0) # 如果文档不在 Qrels 中，相关性为 0
            dcg += relevance / np.log2(rank + 1)
        # else: 无效 hit，跳过

    # --- 计算 IDCG_RunFileDocs@k (基于 Run 数据中文档的理论最高得分) ---
    # 获取 Run File 中所有文档在 Qrels 中的相关性得分
    run_docs_relevances = []
    for doc_id in run_docs_set:
         relevance = query_qrels.get(doc_id, 0)
         run_docs_relevances.append(relevance) # 直接存储相关性得分
    print(f" run_docs_relevances: {run_docs_relevances}")
    # 根据相关性得分降序排序 Run File 中的文档的相关性得分
    # 只考虑相关性大于 0 的文档进行排序构建理想列表的前部分
    ideal_ranked_relevances_runfile = sorted([
        rel for rel in run_docs_relevances if rel > 0
    ], reverse=True)
    
    # 补充 0 相关性的文档直到 k 个位置
    while len(ideal_ranked_relevances_runfile) < k:
         ideal_ranked_relevances_runfile.append(0)
    print(f" ideal_ranked_relevances_runfile: {ideal_ranked_relevances_runfile}")
    # 取前 k 个的相关性得分来计算 IDCG
    ideal_ranked_relevances_runfile_topk = ideal_ranked_relevances_runfile[:k]

    idcg_runfile_docs = 0.0
    for i in range(k):
        idcg_runfile_docs += ideal_ranked_relevances_runfile_topk[i] / np.log2(i + 2) # 位置 i+1 对应 log2(i+2)


    # --- 计算 NDCG ---
    ndcg_runfile_normalized = 0.0
    if idcg_runfile_docs > 0:
        ndcg_runfile_normalized = dcg / idcg_runfile_docs
    # else: 如果 idcg_runfile_docs 是 0 (Run 数据中没有相关文档)，NDCG 就是 0

    return ndcg_runfile_normalized

# --- 示例用法 ---

# 计算 NDCG
# print(run_file[0][])
ndcg_value_dict_auto = calculate_ndcg_runfile_normalized_dict_auto_qid(run_file[0], qrels_dict, k=10)

print(f"NDCG@{10} normalized by IDCG from Run Data Docs (Auto QID) : {ndcg_value_dict_auto:.4f}")

 run_docs_set: {'msmarco_passage_47_609747966', 'msmarco_passage_03_723004579', 'msmarco_passage_55_422869257', 'msmarco_passage_44_853075903', 'msmarco_passage_55_422841976', 'msmarco_passage_59_580811532', 'msmarco_passage_37_36647802', 'msmarco_passage_66_119411167', 'msmarco_passage_65_381245232', 'msmarco_passage_22_349535409', 'msmarco_passage_07_74139026', 'msmarco_passage_55_423408197', 'msmarco_passage_10_630865817', 'msmarco_passage_55_422866120', 'msmarco_passage_53_253592504', 'msmarco_passage_53_592569309', 'msmarco_passage_48_749998273', 'msmarco_passage_48_280793081', 'msmarco_passage_00_546996726', 'msmarco_passage_14_853672414'}
 run_docs_relevances: [1, 0, 0, 0, 0, 3, 1, 1, 0, 0, 1, 2, 0, 0, 1, 2, 2, 0, 2, 2]
 ideal_ranked_relevances_runfile: [3, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1]
NDCG@10 normalized by IDCG from Run Data Docs (Auto QID) : 0.2560


In [4]:
eval_rerank('', run_file[0], qrels_dict)

NameError: name 'eval_rerank' is not defined

In [37]:
import numpy as np
# 不需要 pytrec_eval 库来读取文件了，因为数据已经是 dict 格式

def calculate_standard_ndcg_dict(run_data_dict, qrels_data_dict, k=10):
    """
    计算给定 Run 数据字典的标准 NDCG@k。
    归一化项 (IDCG@k) 基于 Qrels 字典中所有文档的理论最高 DCG@k。
    Query ID 从 run_data_dict['hits'][0]['qid'] 中获取。

    Args:
        run_data_dict (dict): 包含 Run 结果的字典，格式为 {'query': '...', 'hits': [...]}.
                              hits 是一个列表，每个元素是 {'docid': '...', 'qid': ..., 'rank': ..., 'score': ..., ...}。
                              假设 hits 中的所有条目都属于同一个 Query。
        qrels_data_dict (dict): pytrec_eval 格式的 Qrels 字典，格式为 {qid: {docid: relevance, ...}, ...}。
        k (int): 计算的截断位置 (e.g., @10)。

    Returns:
        float: 计算得到的标准 NDCG@k 值。如果无法计算（无 Qrels 判断, Run 数据为空等），返回 0.0。
    """
    all_hits = run_data_dict.get('hits', [])

    if not all_hits:
        # print("Warning: Run data 'hits' list is empty.")
        return 0.0

    # 从第一个 hit 中获取 Query ID
    query_id = all_hits[0].get('qid')
    if query_id is None:
         # print("Warning: 'qid' not found in the first hit.")
         return 0.0


    # 1. 从 Qrels 字典中获取目标 Query 的所有相关性判断
    query_qrels = qrels_data_dict.get(str(query_id))

    if not query_qrels:
        # 没有 Qrels 判断，无法计算 NDCG
        # print(f"Warning: No relevance judgments found for query {query_id} in Qrels dict.")
        return 0.0


    # --- 计算 DCG@k (基于 Run 数据实际排序的前 k 个) ---
    dcg = 0.0
    # 取 hits 列表中的前 k 个 (假设已按 score/rank 排序)
    ranked_hits_for_dcg = all_hits[:k]

    for i in range(len(ranked_hits_for_dcg)):
        hit = ranked_hits_for_dcg[i]
        doc_id = hit.get('docid')
        rank = i + 1 # 位置从 1 开始，使用列表索引+1更通用

        if doc_id:
            # 查找文档在 Qrels 中的真实相关性
            relevance = query_qrels.get(doc_id, 0) # 如果文档不在 Qrels 中，相关性为 0
            dcg += relevance / np.log2(rank + 1)
        # else: 无效 hit，跳过


    # --- 计算 标准 IDCG@k (基于 Qrels 中所有文档的理论最高得分) ---
    # 获取 Qrels 中所有文档的相关性得分
    all_qrels_relevances = list(query_qrels.values())

    # 根据相关性得分降序排序
    ideal_ranked_relevances_qrels = sorted([
        rel for rel in all_qrels_relevances if rel > 0 # 标准 IDCG 只考虑相关性大于 0 的文档进行理想排序的前部分
    ], reverse=True)

    # 补充 0 相关性的文档直到 k 个位置
    while len(ideal_ranked_relevances_qrels) < k:
        ideal_ranked_relevances_qrels.append(0)

    # 取前 k 个的相关性得分来计算标准 IDCG
    ideal_ranked_relevances_qrels_topk = ideal_ranked_relevances_qrels[:k]


    idcg_standard = 0.0
    for i in range(k):
        idcg_standard += ideal_ranked_relevances_qrels_topk[i] / np.log2(i + 2) # 位置 i+1 对应 log2(i+2)


    # --- 计算 标准 NDCG ---
    ndcg_standard = 0.0
    if idcg_standard > 0:
        ndcg_standard = dcg / idcg_standard
    # else: 如果 idcg_standard 是 0 (Qrels 中没有相关文档)，NDCG 就是 0

    return ndcg_standard

# 计算标准 NDCG
standard_ndcg_value = calculate_standard_ndcg_dict(run_file[0], qrels_dict, k=10)

print(f"Standard NDCG@{10} (Normalized by IDCG from ALL Qrels) : {standard_ndcg_value:.4f}")

# 对比一下之前计算的 Normalized by RunFileDocs 的值
# runfile_normalized_ndcg = calculate_ndcg_runfile_normalized_dict_auto_qid(run_data, qrels_dict, k=10)
# print(f"NDCG@{10} Normalized by IDCG from Run Data Docs: {runfile_normalized_ndcg:.4f}")
# 理论上，由于 Standard IDCG 考虑了 Qrels 中所有相关文档（包括 Run Data 中没有的 docU, docV），
# 它通常会 >= IDCG_RunFileDocs。因此，Standard NDCG 结果通常会 <= RunFileNormalized NDCG 结果。

Standard NDCG@10 (Normalized by IDCG from ALL Qrels) : 0.1662


In [45]:
import numpy as np
# 不需要 pytrec_eval 库来读取文件了，因为数据已经是 dict 格式

def calculate_standard_ndcg_dict(run_data_dict, qrels_data_dict, k=10):
    """
    计算给定 Run 数据字典的标准 NDCG@k。
    归一化项 (IDCG@k) 基于 Qrels 字典中所有文档的理论最高 DCG@k。
    Query ID 从 run_data_dict['hits'][0]['qid'] 中获取。

    Args:
        run_data_dict (dict): 包含 Run 结果的字典，格式为 {'query': '...', 'hits': [...]}.
                              hits 是一个列表，每个元素是 {'docid': '...', 'qid': ..., 'rank': ..., 'score': ..., ...}。
                              假设 hits 中的所有条目都属于同一个 Query。
        qrels_data_dict (dict): pytrec_eval 格式的 Qrels 字典，格式为 {qid: {docid: relevance, ...}, ...}。
        k (int): 计算的截断位置 (e.g., @10)。

    Returns:
        float: 计算得到的标准 NDCG@k 值。如果无法计算（无 Qrels 判断, Run 数据为空等），返回 0.0。
    """
    all_hits = run_data_dict.get('hits', [])

    if not all_hits:
        # print("Warning: Run data 'hits' list is empty.")
        return 0.0

    # 从第一个 hit 中获取 Query ID
    query_id = all_hits[0].get('qid')
    if query_id is None:
         # print("Warning: 'qid' not found in the first hit.")
         return 0.0


    # 1. 从 Qrels 字典中获取目标 Query 的所有相关性判断
    query_qrels = qrels_data_dict.get(query_id)

    if not query_qrels:
        # 没有 Qrels 判断，无法计算 NDCG
        # print(f"Warning: No relevance judgments found for query {query_id} in Qrels dict.")
        return 0.0


    # --- 计算 DCG@k (基于 Run 数据实际排序的前 k 个) ---
    dcg = 0.0
    # 取 hits 列表中的前 k 个 (假设已按 score/rank 排序)
    ranked_hits_for_dcg = all_hits[:k]

    for i in range(len(ranked_hits_for_dcg)):
        hit = ranked_hits_for_dcg[i]
        doc_id = hit.get('docid')
        rank = i + 1 # 位置从 1 开始，使用列表索引+1更通用

        if doc_id:
            # 查找文档在 Qrels 中的真实相关性
            relevance = query_qrels.get(doc_id, 0) # 如果文档不在 Qrels 中，相关性为 0
            dcg += relevance / np.log2(rank + 1)
        # else: 无效 hit，跳过


    # --- 计算 标准 IDCG@k (基于 Qrels 中所有文档的理论最高得分) ---
    # 获取 Qrels 中所有文档的相关性得分
    all_qrels_relevances = list(query_qrels.values())

    # 根据相关性得分降序排序
    ideal_ranked_relevances_qrels = sorted([
        rel for rel in all_qrels_relevances if rel > 0 # 标准 IDCG 只考虑相关性大于 0 的文档进行理想排序的前部分
    ], reverse=True)

    # 补充 0 相关性的文档直到 k 个位置
    while len(ideal_ranked_relevances_qrels) < k:
        ideal_ranked_relevances_qrels.append(0)

    # 取前 k 个的相关性得分来计算标准 IDCG
    ideal_ranked_relevances_qrels_topk = ideal_ranked_relevances_qrels[:k]


    idcg_standard = 0.0
    for i in range(k):
        idcg_standard += ideal_ranked_relevances_qrels_topk[i] / np.log2(i + 2) # 位置 i+1 对应 log2(i+2)


    # --- 计算 标准 NDCG ---
    ndcg_standard = 0.0
    if idcg_standard > 0:
        ndcg_standard = dcg / idcg_standard
    # else: 如果 idcg_standard 是 0 (Qrels 中没有相关文档)，NDCG 就是 0

    return ndcg_standard

# --- 示例用法 ---

# 使用之前定义的模拟 Qrels 字典 (包含 docU, docV)
qrels_dict = {
    '646091': {'docA': 3, 'docB': 2, 'docC': 1, 'docD': 0, 'docE': 3, 'docF': 2,
             'docG': 1, 'docH': 0, 'docI': 3, 'docJ': 2, 'docK': 1, 'docL': 0,
             'docM': 3, 'docN': 2, 'docO': 1, 'docP': 0, 'docQ': 3, 'docR': 2,
             'docS': 1, 'docT': 0, 'docU': 3, 'docV': 2} # 包含 docU(3) 和 docV(2)
}

# 使用之前定义的模拟 Run 数据字典 (包含前 20 个文档的重排序)
run_data = {
    'query': 'what does prenatal care include',
    'hits': [
        {'content': '...', 'docid': 'docD', 'qid': 646091, 'rank': 1, 'score': 0.9},
        {'content': '...', 'docid': 'docH', 'qid': 646091, 'rank': 2, 'score': 0.8},
        {'content': '...', 'docid': 'docT', 'qid': 646091, 'rank': 3, 'score': 0.7},
        {'content': '...', 'docid': 'docL', 'qid': 646091, 'rank': 4, 'score': 0.6},
        {'content': '...', 'docid': 'docP', 'qid': 646091, 'rank': 5, 'score': 0.5},
        {'content': '...', 'docid': 'docA', 'qid': 646091, 'rank': 6, 'score': 0.4},
        {'content': '...', 'docid': 'docM', 'qid': 646091, 'rank': 7, 'score': 0.3},
        {'content': '...', 'docid': 'docI', 'qid': 646091, 'rank': 8, 'score': 0.2},
        {'content': '...', 'docid': 'docQ', 'qid': 646091, 'rank': 9, 'score': 0.1},
        {'content': '...', 'docid': 'docE', 'qid': 646091, 'rank': 10, 'score': 0.05},
        {'content': '...', 'docid': 'docB', 'qid': 646091, 'rank': 11, 'score': 0.04},
        {'content': '...', 'docid': 'docF', 'qid': 646091, 'rank': 12, 'score': 0.03},
        {'content': '...', 'docid': 'docJ', 'qid': 646091, 'rank': 13, 'score': 0.02},
        {'content': '...', 'docid': 'docN', 'qid': 646091, 'rank': 14, 'score': 0.01},
        {'content': '...', 'docid': 'docR', 'qid': 646091, 'rank': 15, 'score': 0.009},
        {'content': '...', 'docid': 'docC', 'qid': 646091, 'rank': 16, 'score': 0.008},
        {'content': '...', 'docid': 'docG', 'qid': 646091, 'rank': 17, 'score': 0.007},
        {'content': '...', 'docid': 'docK', 'qid': 646091, 'rank': 18, 'score': 0.006},
        {'content': '...', 'docid': 'docO', 'qid': 646091, 'rank': 19, 'score': 0.005},
        {'content': '...', 'docid': 'docS', 'qid': 646091, 'rank': 20, 'score': 0.004},
    ]
}

# 计算标准 NDCG
standard_ndcg_value = calculate_standard_ndcg_dict(run_data, qrels_dict, k=10)

print(f"Standard NDCG@{10} (Normalized by IDCG from ALL Qrels) : {standard_ndcg_value:.4f}")

# 对比一下之前计算的 Normalized by RunFileDocs 的值
# runfile_normalized_ndcg = calculate_ndcg_runfile_normalized_dict_auto_qid(run_data, qrels_dict, k=10)
# print(f"NDCG@{10} Normalized by IDCG from Run Data Docs: {runfile_normalized_ndcg:.4f}")
# 理论上，由于 Standard IDCG 考虑了 Qrels 中所有相关文档（包括 Run Data 中没有的 docU, docV），
# 它通常会 >= IDCG_RunFileDocs。因此，Standard NDCG 结果通常会 <= RunFileNormalized NDCG 结果。

Standard NDCG@10 (Normalized by IDCG from ALL Qrels) : 0.0000


In [49]:
k_values=[10]
ndcg = {f"NDCG@{k}": 0.0 for k in k_values}
_map = {f"MAP@{k}": 0.0 for k in k_values}
recall = {f"Recall@{k}": 0.0 for k in k_values}
map_string = "map_cut." + ",".join(map(str, k_values))
ndcg_string = "ndcg_cut." + ",".join(map(str, k_values))
recall_string = "recall." + ",".join(map(str, k_values))
evaluator = pytrec_eval.RelevanceEvaluator(qrels_dict, {map_string, ndcg_string, recall_string})
scores = evaluator.evaluate(run_file[0])
scores


TypeError: Unable to extract query/object scores.

In [2]:
import pytrec_eval
import numpy as np # 依然需要 numpy for log2, although pytrec_eval handles it internally

# 使用之前定义的模拟 Qrels 字典 (包含 docU, docV)
qrels_dict = {
    '646091': {'docA': 3, 'docB': 2, 'docC': 1, 'docD': 0, 'docE': 3, 'docF': 2,
             'docG': 1, 'docH': 0, 'docI': 3, 'docJ': 2, 'docK': 1, 'docL': 0,
             'docM': 3, 'docN': 2, 'docO': 1, 'docP': 0, 'docQ': 3, 'docR': 2,
             'docS': 1, 'docT': 0, 'docU': 3, 'docV': 2} # 包含 docU(3) 和 docV(2)
}

# 使用之前定义的模拟 Run 数据字典 (包含前 20 个文档的重排序)
run_data = {
    'query': 'what does prenatal care include', # 这个字段在评测时不用，qid 字段是关键
    'hits': [
        {'content': '...', 'docid': 'docD', 'qid': 646091, 'rank': 1, 'score': 0.9},
        {'content': '...', 'docid': 'docH', 'qid': 646091, 'rank': 2, 'score': 0.8},
        {'content': '...', 'docid': 'docT', 'qid': 646091, 'rank': 3, 'score': 0.7},
        {'content': '...', 'docid': 'docL', 'qid': 646091, 'rank': 4, 'score': 0.6},
        {'content': '...', 'docid': 'docP', 'qid': 646091, 'rank': 5, 'score': 0.5},
        {'content': '...', 'docid': 'docA', 'qid': 646091, 'rank': 6, 'score': 0.4},
        {'content': '...', 'docid': 'docM', 'qid': 646091, 'rank': 7, 'score': 0.3},
        {'content': '...', 'docid': 'docI', 'qid': 646091, 'rank': 8, 'score': 0.2},
        {'content': '...', 'docid': 'docQ', 'qid': 646091, 'rank': 9, 'score': 0.1},
        {'content': '...', 'docid': 'docE', 'qid': 646091, 'rank': 10, 'score': 0.05},
        {'content': '...', 'docid': 'docB', 'qid': 646091, 'rank': 11, 'score': 0.04},
        {'content': '...', 'docid': 'docF', 'qid': 646091, 'rank': 12, 'score': 0.03},
        {'content': '...', 'docid': 'docJ', 'qid': 646091, 'rank': 13, 'score': 0.02},
        {'content': '...', 'docid': 'docN', 'qid': 646091, 'rank': 14, 'score': 0.01},
        {'content': '...', 'docid': 'docR', 'qid': 646091, 'rank': 15, 'score': 0.009},
        {'content': '...', 'docid': 'docC', 'qid': 646091, 'rank': 16, 'score': 0.008},
        {'content': '...', 'docid': 'docG', 'qid': 646091, 'rank': 17, 'score': 0.007},
        {'content': '...', 'docid': 'docK', 'qid': 646091, 'rank': 18, 'score': 0.006},
        {'content': '...', 'docid': 'docO', 'qid': 646091, 'rank': 19, 'score': 0.005},
        {'content': '...', 'docid': 'docS', 'qid': 646091, 'rank': 20, 'score': 0.004},
    ]
}

# 确定要评测的 Query ID
query_id = str(run_data['hits'][0]['qid']) # 确保 QID 是字符串类型以匹配 qrels_dict 的键


# # 1. 加载 Qrels 数据到 RelevanceEvaluator
# # pytrec_eval.RelevanceEvaluator 可以直接接受 pytrec_eval dict 格式的 Qrels
evaluator = pytrec_eval.RelevanceEvaluator(qrels_dict, {f'ndcg.{10}'}) # 指定要计算的指标是 ndcg@10

# # 2. 准备 Run 数据格式
# # pytrec_eval 的 evaluate 方法需要 Run 数据是 {query_id: {doc_id: score, ...}, ...} 格式
# # 注意：这里只需要 doc_id 和 score，rank 和 content 等字段不需要给 evaluator
run_data_formatted_for_evaluator = {}
run_data_formatted_for_evaluator[query_id] = {}

# # 遍历 run_data['hits'] 列表，将每个 hit 转换为 evaluator 需要的格式
for hit in run_data['hits']:
    doc_id = hit.get('docid')
    score = hit.get('score')
    if doc_id is not None and score is not None:
        run_data_formatted_for_evaluator[query_id][doc_id] = score

# # 3. 进行评测
# # evaluate 方法返回一个嵌套字典 {query_id: {metric_name: value, ...}, ...}
# results = evaluator.evaluate(run_data_formatted_for_evaluator)

# # 4. 获取并打印结果
# if query_id in results and f'ndcg.{10}' in results[query_id]:
#      pytrec_eval_ndcg_value = results[query_id][f'ndcg.{10}']
#      print(f"pytrec_eval standard NDCG@{10} for query {query_id}: {pytrec_eval_ndcg_value:.4f}")
# else:
#      print(f"pytrec_eval could not calculate NDCG@{10} for query {query_id}.")

# # 我们可以对比一下之前我们函数计算的 standard NDCG
# from your_code import calculate_standard_ndcg_dict # 假设你的函数在你 import your_code 的地方

# my_standard_ndcg = calculate_standard_ndcg_dict(run_data, qrels_dict, k=10)
# print(f"My code's standard NDCG@{10} for query {query_id}: {my_standard_ndcg:.4f}")

# # 现在你可以比较这两个结果，看看它们是否一致或非常接近。
# 如果差异较大，如我们之前讨论，很可能是 Qrels 数据、Run 数据或 pytrec_eval 配置与预期不符。

In [3]:
run_data_formatted_for_evaluator

{'646091': {'docD': 0.9,
  'docH': 0.8,
  'docT': 0.7,
  'docL': 0.6,
  'docP': 0.5,
  'docA': 0.4,
  'docM': 0.3,
  'docI': 0.2,
  'docQ': 0.1,
  'docE': 0.05,
  'docB': 0.04,
  'docF': 0.03,
  'docJ': 0.02,
  'docN': 0.01,
  'docR': 0.009,
  'docC': 0.008,
  'docG': 0.007,
  'docK': 0.006,
  'docO': 0.005,
  'docS': 0.004}}